# GEMs Project 06/08 - 06/12 Notebook

```
Continuing work from last week's notebook:
/home/gmvaz/2026_GEMs/stm_model_test/0601_0605.ipynb
```
Models this week:

control - `STM_V1_0`

stm1090 - generated from...

stm1585 - 

stm2575 - 

## Goal
Create a medium with anaerobic conditions and the metabolite conditions SW emailed me


## Objectives
1. How much oxygen is available to Salmonella in the gut during infection?
   - Is it fully anaerobic or is there "some" oxygen?
   - How do we quantify that to a reaction flux rate?
2. How do I create a new `medium` for the model?
   - Create a medium with metabolites to simulate an inflamed gut environment. 

## Tasks
1. Simulate a range of anaerobic conditions
2. Find out the context for the medium conditions from STM_V1_0
3. Create a new medium for the metabolite conditions Sebastian emailed (or alter the existing medium)
4. Calculate minimal media

### Pending Questions: 
1. `.slim_optimize()` vs `.optimize()`?
2. What are boundary reactions? Demand, sink, exchange?

## Generate Models

In [1]:
import os
from cobra.core.configuration import Configuration

# Set the Gurobi license and solver
os.environ["GRB_LICENSE_FILE"] = "/home/gmvaz/.gurobi.lic"
Configuration().solver = "gurobi"

In [2]:
from cobra.io import load_model, load_json_model

# load the test S.Tm LT2 GEM in cobrapy
control_model = load_model("STM_v1_0")

Set parameter WLSAccessID
Set parameter WLSSecret
Set parameter LicenseID to value 2789960
Academic license 2789960 - for non-commercial use only - registered to gm___@ucdavis.edu


In [3]:
import pandas as pd

# read in the gene weights csv files of the different threshold rna-seq data
stm_threshold_10_90 = pd.read_csv("/home/gmvaz/2026_GEMs/imat_prep_test/stm_threshold_10_90.csv")
stm_threshold_15_85 = pd.read_csv("/home/gmvaz/2026_GEMs/imat_prep_test/stm_threshold_15_85.csv")
stm_threshold_25_75 = pd.read_csv("/home/gmvaz/2026_GEMs/imat_prep_test/stm_threshold_25_75.csv")

In [4]:
# make each list of gene weights into a python set
stm_set_10_90 = set(stm_threshold_10_90["genes"].astype(str).str.strip())
stm_set_15_85 = set(stm_threshold_15_85["genes"].astype(str).str.strip())
stm_set_25_75 = set(stm_threshold_25_75["genes"].astype(str).str.strip())

# check the length
print(len(stm_set_10_90))
print(len(stm_set_15_85))
print(len(stm_set_25_75))

4671
4671
4671


In [5]:
# make a list of all genes in the control model and save it as a set
model_genes = set([g.id for g in control_model.genes])

#check
print(len(model_genes))

1271


In [6]:
# subset the gene weights list for each model/sample to only have the genes present in the input model

stm1090_model = stm_threshold_10_90[stm_threshold_10_90["genes"].isin(model_genes)]
stm1585_model = stm_threshold_15_85[stm_threshold_15_85["genes"].isin(model_genes)]
stm2575_model = stm_threshold_25_75[stm_threshold_25_75["genes"].isin(model_genes)]

In [7]:
# there are different amounts of genes in the gene weights lists and in the input model gene list. 
# make a new gene weight list per sample that has all the genes from the input model:
# (1) assign a value of 0 to the 20 genes found in the input model but not in the gene weights list. 
# (2) add the genes that are present in the input model and raw gene weight list. 
# Result: a gene weight list per sample that has only the genes found in the input model and raw gene weights list.

subset_weights_stm1090 = stm1090_model.set_index("genes")["threshold_category"].reindex(model_genes, fill_value=0)
subset_weights_stm1585 = stm1585_model.set_index("genes")["threshold_category"].reindex(model_genes, fill_value=0)
subset_weights_stm2575 = stm2575_model.set_index("genes")["threshold_category"].reindex(model_genes, fill_value=0)

# check
print(len(subset_weights_stm1090))
print(len(subset_weights_stm1585))
print(len(subset_weights_stm2575))

1271
1271
1271


In [8]:
from imatpy.parse_gpr import gene_to_rxn_weights

# convert gene weights to reaction weights using the final gene weights lists with all genes from the input model
rxn_weights_stm1090 = gene_to_rxn_weights(control_model, subset_weights_stm1090)
rxn_weights_stm1585 = gene_to_rxn_weights(control_model, subset_weights_stm1585)
rxn_weights_stm2575 = gene_to_rxn_weights(control_model, subset_weights_stm2575)

# check
print(len(rxn_weights_stm1090))
print(len(rxn_weights_stm1585))
print(len(rxn_weights_stm2575))

2545
2545
2545


In [9]:
from imatpy.model_creation import generate_model

# generate context-specific models with imat by integrating the reaction weights
stm1090_model = generate_model(model = control_model, rxn_weights = rxn_weights_stm1090, method = "imat_restrictions")
stm1585_model = generate_model(model = control_model, rxn_weights = rxn_weights_stm1585, method = "imat_restrictions")
stm2575_model = generate_model(model = control_model, rxn_weights = rxn_weights_stm2575, method = "imat_restrictions")

Read LP format model from file /tmp/tmpr6f4kggk.lp
Reading time = 0.02 seconds
: 1802 rows, 5090 columns, 19612 nonzeros
Read LP format model from file /tmp/tmpeuemvgcy.lp
Reading time = 0.02 seconds
: 1802 rows, 5090 columns, 19612 nonzeros
Read LP format model from file /tmp/tmpjgcez5f9.lp
Reading time = 0.02 seconds
: 1802 rows, 5090 columns, 19612 nonzeros


In [10]:
# copy the models to run FBA 
control_model_fba = load_model("STM_v1_0")
stm1090_model_fba = stm1090_model.copy()
stm1585_model_fba = stm1585_model.copy()
stm2575_model_fba = stm2575_model.copy()

# run FBA on all models
control_model_fba_res = control_model_fba.optimize()
stm1090_fba_res = stm1090_model_fba.optimize()
stm1585_fba_res = stm1585_model_fba.optimize()
stm2575_fba_res = stm2575_model_fba.optimize()

Read LP format model from file /tmp/tmpr5w8ivuv.lp
Reading time = 0.02 seconds
: 2947 rows, 6220 columns, 24174 nonzeros
Read LP format model from file /tmp/tmps8028ipi.lp
Reading time = 0.02 seconds
: 3201 rows, 6460 columns, 25176 nonzeros
Read LP format model from file /tmp/tmpup1tf1cl.lp
Reading time = 0.02 seconds
: 3949 rows, 7128 columns, 28088 nonzeros


In [11]:
# print the objective value for each model after FBA

print(control_model_fba_res.objective_value)
print(stm1090_fba_res.objective_value)
print(stm1585_fba_res.objective_value)
print(stm2575_fba_res.objective_value)

0.4778336607607207
0.4072058061353676
0.40650099245760307
0.3934219732489797


## Copy Default Medium and Change to Anaerobic

In [12]:
# look at the medium for the control_model

control_model_fba.medium

{'EX_ca2_e': 1000.0,
 'EX_cbl1_e': 0.01,
 'EX_cl_e': 1000.0,
 'EX_co2_e': 1000.0,
 'EX_cobalt2_e': 1000.0,
 'EX_cu2_e': 1000.0,
 'EX_mg2_e': 1000.0,
 'EX_fe2_e': 1000.0,
 'EX_fe3_e': 1000.0,
 'EX_mn2_e': 1000.0,
 'EX_mobd_e': 1000.0,
 'EX_na1_e': 1000.0,
 'EX_nh4_e': 1000.0,
 'EX_o2_e': 18.5,
 'EX_pi_e': 1000.0,
 'EX_zn2_e': 1000.0,
 'EX_so4_e': 1000.0,
 'EX_glc__D_e': 5.0,
 'EX_tungs_e': 1000.0,
 'EX_h2o_e': 100.0,
 'EX_h_e': 100.0,
 'EX_k_e': 1000.0}

In [13]:
# create a new variable, no_oxygen_control, and load/assign the STM_v1_0 model to it
no_oxygen_control = load_model("STM_v1_0")

# create a no_oxygen_medium variable and assign the medium from the no_oxygen_control model to it
no_oxygen_medium = no_oxygen_control.medium

# change the upper bound of the oxygen exchange reaction to 0 in the no_oxygen_control medium 
no_oxygen_medium["EX_o2_e"] = 0.0

# assign no_oxygen_medium to the no_oxygen_control model 
no_oxygen_control.medium = no_oxygen_medium

## How to create a new medium with exchange reactions of interest?

Based on [this](https://cobrapy.readthedocs.io/en/latest/media.html) section of the documentation, I know that the medium of a model can be altered by copying the `medium` dictionary, altering the exchange reaction of interest, and assigning the new medium dictionary to the model I want to simulate. 

What I want to do:

I want to create a medium with exchange reactions associated with the metabolites list I have from SW's email and assign it to the models to simulate growth. 

What I know so far.. 06/08/2026

The medium attribute returns a dictionary of the upper flux bounds for all active exchange reactions. The `medium` attribute contains the reactions that control the availability of nutrients to a model - in other words, it manages the experimental conditions or "growth medium" of the model. To change the medium, I copy it, save it as a new dictionary, alter the new dictionary, and assign the new dictionary to a model. The "Growth Media" portion of the docs does not elaborate on how to add exchange reactions to the medium which leads me to think that I need to create a dictionary with the exchange reactions that correspond with Sebastian's list to have a `medium` attribute that reflects the environment I want to simulate. 

This brings up a lot of questions though. I find it odd that I haven't read of a method to add exchange reactions to the medium? I wonder if this is something specified in the docs and I just haven't realized it yet. If so, I'm not sure how to "look" for this. Generally speaking, I'm confused on how to get more information on the `medium` attribute either through the documentation or by using Python commands. I know that `medium` is an attribute of the model but how do I know the functions to "get" the exchange reactions I need and copy them into a dictionary? If there is sych a method, where/how would I look for it? 

## Sebastian's List of Metabolites - 06/12/26

O2: ~1%

Tetrathionate: accumulates to up to ~4 mM in Salmonella-infected mice => 4mM  PMID: 20864996

Nitrate: 0.4 mM nitrate in DSS model, PMID:23393266, similar in Salmonella-infected mice 26099579. => 0.5 mM

Ethanolamine: ~1 mM in Salmonella infected mice  PMID: 21969563 => 1 mM

1,2 propanediol:  > 40 uM in B theta-associated GF mice infected with Salmonella, PMID:28056091 => 0.1 mM

L-Lactate: up to 8.5 mM in strep-treated mice PMID: 29276172. => 10 mM

Formate: accumulates to up to ~4 mM in Salmonella-infected mice 37498116 => 4 mM

Fructose-Asp: ? In diet, between 1 and 10000 umol/kg (~10 mM) 29232127. 10 pmol/mg (~10 uM) in uninfected or Salmonella-infected mice 29483291 => 0.1 mM

Succinate: up to 3 mM in the colon content of Salmonella-infected mice PMID:28844888. => 3 mM

Aspartate: 1–5 mM   PMID:32416061 uninfected mice  => 5 mM

Malate: 100–200 μM PMID:32416061 uninfected mice => 0.2 mM

Galactarate: 10 ng/mg (10 mM) in strep-treated mice PMID:27309805 => 10 mM

Glucarate: up to 80 ng/mg (80 mM, although there was a lot of variation) in strep-treated mice PMID:27309805 => 10 mM

Pyruvate: 90 umol/mg (90 mmol/g = 90 mol/l = unrealistic, probably wrong in the paper!!!!) in infected mice => 10 mM PMID: 34349263

Arabinose: 2 ug/mg PMID 36812913

SCFA (at steady state):

Butyrate: 10 mM

Acetate: 50 mM

Propionate: 5 mM

## Make a Medium with SW's Metabolite List

In [ ]:
# Make a new, empty dictionary
# Get a list of the metabolites and their concentration
# Find the exchange reaction for that metabolite
# Add exchange reaction to the dictionary
# Change the metabolite concentration 

sw_medium = {
    

}